# SymCity codes

`SymCityCode` constructs self-dual polynomial CSS codes with prescribed regular logical
actions of a finite abelian group `G` and cyclic symmetry among physical types. The
coefficient ring specifies `G`; the physical cycle of length `q` is a separate choice.
`block_coefficients[a][b][d]` specifies power `d` of the physical-cycle variable, for
check family `a` and physical cycle `b`. There can be any supported number of families
and cycles; the input is a rectangular `p × t × q` coefficient tensor.

Supply the check polynomials and the constructor resolves an independent basis of
complete `G` orbits, then derives a compatible common X/Z logical frame. Both stages
validate their results. A supplied presentation may have redundant checks, but an
orbit basis or a compatible logical frame need not exist for every commuting input.
Construction does not certify distance or optimize check weights.

## Redundant families over a noncyclic group

Here `G = C_2 × C_2`, `p = 3`, `t = 5`, and `q = 1`. The third prototype is the sum
of the first two. The general rank test finds the independent orbits automatically.
This constant-coefficient example consists of copies indexed by `G`.

In [1]:
import numpy as np

from qldpc import abstract, codes
from qldpc.objects import Pauli

ring = abstract.GroupRing(abstract.AbelianGroup(2, 2))
coefficients = abstract.RingArray.build(
    [[[1], [1], [0], [0], [0]], [[0], [0], [1], [1], [0]], [[1], [1], [1], [1], [0]]],
    ring=ring,
)
code = codes.SymCityCode(coefficients)
assert (code.num_qubits, code.dimension) == (20, 4)
assert code.check_basis_result.selected_rows == (0, 1)
assert code.check_basis_result.method == "greedy"
print(code)
print(code.check_basis_result.message)
print(code.logical_frame_result.message)

SymCityCode on 20 qubits with 5 physical cycles of length 1, 1 logical orbits, and logical group AbelianGroup(2,2)
Found 2 independent check orbits (greedy).
Found a compatible J after testing 1 coefficient candidates.


## Basis resolution and its limits

The fast path adds a prototype only if its full orbit increases binary rank by `|G|`.
An orbit with a smaller nonzero contribution is not redundant: a different selection
or polynomial combination may be needed. The fallback tries alternative parent subsets,
then combinations from the full check space. The latter is exhaustive when unbounded
and has exponential worst-case cost.

`check_search_limit` bounds fallback rank trials; greedy selection remains enabled at
zero. The separate `logical_search_limit` bounds logical-frame coefficient candidates.
Neither is a wall-clock limit. Each defaults to 10,000; `None` allows exhaustion.

A successful constructor exposes `check_basis_result` and `logical_frame_result`.
`CheckBasisError.result` reports a basis failure; `LogicalFrameError.result` reports a
logical-frame failure. `infeasible` means a proof of impossibility for that requirement,
whereas `inconclusive` means the configured budget was insufficient.

### A basis that needs polynomial combinations

In `F_2[C_3]`, the two complementary idempotents below each generate only part of a
regular orbit. Neither displayed row can be an independent orbit basis; their sum can.

In [2]:
ring = abstract.GroupRing(abstract.CyclicGroup(3))
x = ring.generators[0]
e0, e1 = ring.one + x + x**2, x + x**2
coefficients = abstract.RingArray.build([[[e0], [e0], [0]], [[e1], [e1], [0]]], ring)
combined = codes.SymCityCode(coefficients)
assert combined.check_basis_result.method == "combinations"
assert combined.check_basis_result.selected_rows is None
assert (len(combined), combined.dimension) == (9, 3)
print(combined.check_basis_result.message)

try:
    codes.SymCityCode(coefficients, check_search_limit=0)
except codes.CheckBasisError as error:
    assert error.result.status == "inconclusive"
    print(error.result.status, error.result.message)

# Explicit validated bases and frames can be reused without either fallback search.
reused = codes.SymCityCode(
    coefficients,
    combined.logical_prototypes,
    check_basis=combined.check_basis,
    check_search_limit=0,
    logical_search_limit=0,
)
assert reused == combined

Found 1 independent check orbits using polynomial combinations.
inconclusive No complete independent check-orbit basis found within 0 fallback trials; existence remains undecided. Increase check_search_limit or supply check_basis.


### A check space without an independent orbit basis

Here the binary rank is divisible by `|G|`, but every check is fixed by the nonidentity
translation. Exhausting the small full check space proves that no independent regular
orbit basis exists. The commuting parent still defines an ordinary CSS code.

In [3]:
ring = abstract.GroupRing(abstract.CyclicGroup(2))
u = ring.one + ring.generators[0]
nonfree = abstract.RingArray.build([[[u], [0], [0]], [[0], [u], [0]]], ring)
try:
    codes.SymCityCode(nonfree, check_search_limit=None)
except codes.CheckBasisError as error:
    assert error.result.status == "infeasible"
    print(error.result.status, error.result.message)

infeasible No complete independent G-orbit basis exists: exhausted the full check-module domain. The check module is not free; this does not invalidate the parent CSS checks.


## A bicycle-chain example

The same general constructor also accepts this two-block presentation. Its redundant
check orbit is removed automatically, and the existing fiber shortcut supplies the
logical frame. Neither operation requires a named-code special case.

In [4]:
ring = abstract.GroupRing(abstract.CyclicGroup(4))
x = ring.generators[0]
zero, one = ring.zero, ring.one
q = 7
# a(x,y) = 1 + y**3 + x*y**2 + x*y**4
# Coefficient index is the exponent of y, the physical-cycle variable.
a = [one, zero, x, one, x, zero, zero]
a_dagger = [a[(-d) % q].T for d in range(q)]
star = codes.SymCityCode([[a, a_dagger]])
assert (star.num_qubits, star.dimension, star.get_weight()) == (56, 8, 8)
print(star)
print(star.logical_frame_result.message)

SymCityCode on 56 qubits with 2 physical cycles of length 7, 2 logical orbits, and logical group CyclicGroup
Found a compatible J with 2 logical orbits (fiber shortcut).


## Coordinates and explicit logical representatives

Physical columns have order `(cycle, position, group_index)` with group index varying fastest.
Group indices follow `tuple(code.group.generate())`. Logical rows have order
`(logical_orbit, group_index)`. Translation in `G` permutes each logical orbit regularly.

The independent checks span the entire cyclic parent. Dropping rows here removes redundancy;
it does not omit stabilizers as an active-row GALA construction can do. The additional physical
cycle preserves the stabilizer space, even if it does not permute the selected rows as a list.
Its logical action is not assumed to be trivial in the general family.

An explicit polynomial `J` selects particular representatives and bypasses automatic solving.
Its coefficients must use the same ring as the checks.


In [5]:
j = abstract.RingArray.build([[1] * q + [0] * q, [0] * q + [1] * q], ring)
explicit = codes.SymCityCode([[a, a_dagger]], logical_ops=j, check_rows=range(q - 1))
assert np.array_equal(explicit.get_logical_ops(Pauli.X), j.regular_lift())
assert np.array_equal(explicit.get_logical_ops(Pauli.Z), j.regular_lift())
print(explicit.logical_frame_result.message)

Validated the supplied compatible J.


## One physical cycle and a noncyclic logical group

This seven-position polynomial produces one regular logical orbit. For constant coefficients,
the code consists of independent Steane copies, one at each group site. The use of the regular
representation is deliberate, including when the supplied group has a different default lift.


In [6]:
group = abstract.AbelianGroup(2, 2, direct_sum=True)
ring = abstract.GroupRing(group)
coefficients = abstract.RingArray.build([[[1, 1, 1, 0, 1, 0, 0]]], ring)
steane_copies = codes.SymCityCode(coefficients)
assert (len(steane_copies), steane_copies.dimension) == (28, 4)
print(steane_copies)

SymCityCode on 28 qubits with 1 physical cycles of length 7, 1 logical orbits, and logical group AbelianGroup(2,2)


## A general coefficient solve and its outcomes

The fiber shortcut is not applicable to the following `q=1` example. The general solver reduces
check/logical commutation and augmentation to an affine binary system, then checks orthonormality
on its coefficient assignments. This reference method is exponential. Its default limit of
10,000 candidates bounds enumeration, not wall-clock time.

Success yields `found`. Complete exhaustion or a proved obstruction yields `infeasible`.
Stopping before exhausting the domain yields `inconclusive`; that is not a nonexistence proof.
Failures raise `LogicalFrameError`, with the structured outcome available as `.result`.


In [7]:
ring = abstract.GroupRing(abstract.CyclicGroup(3))
x = ring.generators[0]
blocks = [[[ring.one], [x], [ring.zero]]]
solved = codes.SymCityCode(blocks)
assert (len(solved), solved.dimension) == (9, 3)
assert solved.logical_frame_result.method == "coefficients"
print(solved.logical_frame_result.message)

try:
    codes.SymCityCode(blocks, logical_search_limit=0)
except codes.LogicalFrameError as error:
    assert error.result.status == "inconclusive"
    print(error.result.status, error.result.message)

# An explicit frame succeeds even when enumeration is disabled.
j = abstract.RingArray.build([[ring.one, x, ring.one]], ring)
assert codes.SymCityCode(blocks, j, logical_search_limit=0).dimension == 3

Found a compatible J after testing 1 coefficient candidates.
inconclusive No compatible J found within 0 candidates; existence remains undecided. Increase logical_search_limit or supply logical_ops explicitly.


In [8]:
binary_ring = abstract.GroupRing(abstract.TrivialGroup())
impossible = abstract.RingArray.build([[[1, 1, 1, 1]]], binary_ring)
try:
    codes.SymCityCode(impossible, logical_search_limit=None)
except codes.LogicalFrameError as error:
    assert error.result.status == "infeasible"
    print(error.result.status, error.result.message)

infeasible No compatible J exists; all candidates in the affine solution space failed.


These last checks still define an ordinary CSS code, but all commuting supports have even
weight, so an orthonormal common logical frame cannot exist. The constructor does not silently
substitute a generic binary logical basis.

## Overlapping frames and an explicit check basis

Polynomial row combinations can replace selected parent rows when they span exactly the same
check module. Logical representatives may also be changed by stabilizers while preserving all
required identities, including augmentation. The example below creates overlap between translates.


In [9]:
ring = abstract.GroupRing(abstract.CyclicGroup(4))
x = ring.generators[0]
coefficients = abstract.RingArray.build([[[1, 1, 1, 0, 1, 0, 0]]], ring)
base = codes.SymCityCode(coefficients)
h = abstract.RingArray.build(base.check_basis, ring)
h[0] = h[0] + x * h[1]
j = abstract.RingArray.build(base.logical_prototypes, ring)
j[0] = j[0] + (ring.one + x + x**2 + x**3) * base.check_basis[0]
changed = codes.SymCityCode(coefficients, j, check_basis=h)
assert np.array_equal(changed.matrix_x.row_space(), base.matrix_x.row_space())
assert np.any(np.logical_and(*changed.get_logical_ops(Pauli.X)[:2]))
print(changed.logical_frame_result.message)

Validated the supplied compatible J.
